# Demo 07 - eSSVI calibration, smooth projection, and Dupire-oriented handoff

This notebook calibrates eSSVI nodes on the shared quote set, projects them to a smoother implied surface, compares quote fit and expiry-direction smoothness against the slice-wise SVI alternative, and builds the implied object used for local-vol extraction.

- calibrate the nodal eSSVI surface
- project to a smoothed eSSVI surface
- compare quote fit before and after smoothing
- compare expiry-direction smoothness for the Dupire handoff

`PROFILE = "quick"` keeps the bridge compact; switch to `PROFILE = "full"` for denser diagnostics.

In [ ]:
# ruff: noqa: E402
from __future__ import annotations

import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'option_pricing').exists():
            return candidate
    return start

ROOT = _find_repo_root(Path.cwd())
SRC = ROOT / 'src'
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROFILE = 'quick'
SEED = 7

,profile,projection_success,dupire_ready_surface,localvol_type,next_demo
0,quick,True,ESSVISmoothedSurface,LocalVolSurface,08_localvol_pde_repricing


In [ ]:
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from IPython.display import display  # noqa: E402

from option_pricing.demos import (  # noqa: E402
    build_essvi_bridge_artifacts,
    build_shared_demo_scenario,
    build_surface_demo_artifacts,
)

pd.set_option('display.max_columns', None)

scenario = build_shared_demo_scenario(profile=PROFILE, seed=SEED)
surface_art = build_surface_demo_artifacts(
    profile=PROFILE,
    seed=SEED,
    scenario=scenario,
)
art = build_essvi_bridge_artifacts(
    profile=PROFILE,
    seed=SEED,
    scenario=scenario,
    surface_artifacts=surface_art,
)

headline = pd.DataFrame(
    [
        {
            'profile': art.profile,
            'projection_success': art.meta['projection_success'],
            'dupire_ready_surface': art.meta['dupire_ready_surface'],
            'localvol_type': type(art.localvol).__name__,
            'next_demo': '08_localvol_pde_repricing',
        }
    ]
)
display(headline)

## 1) Calibrate eSSVI nodes and project the smooth surface

The nodal fit and projection are kept separate so the calibration inputs and the smoother downstream object can both be inspected directly.

In [2]:
display(art.tables['essvi_projection_summary'])
display(art.tables['essvi_nodes'])

quotes = art.tables['quotes_df'].copy()
Ts = np.unique(quotes['T'].to_numpy(dtype=float))
idx = np.unique(np.linspace(0, len(Ts) - 1, min(3, len(Ts)), dtype=int))
T_show = [float(Ts[i]) for i in idx]
y_curve = np.linspace(-0.28, 0.28, 121)

fig, axes = plt.subplots(1, len(T_show), figsize=(5 * len(T_show), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, T in zip(axes, T_show, strict=True):
    q = quotes.loc[np.isclose(quotes['T'], T)].sort_values('K')
    F = float(art.scenario.forward(T))
    K_curve = F * np.exp(y_curve)
    ax.scatter(q['K'], q['iv_obs'], s=22, alpha=0.75, label='quotes')
    ax.plot(K_curve, art.nodal_surface.iv(y_curve, T), linewidth=2.0, label='eSSVI nodes')
    ax.plot(K_curve, art.smoothed_surface.iv(y_curve, T), linewidth=2.0, linestyle='--', label='smoothed')
    ax.set_title(f'T={T:g}')
    ax.set_xlabel('Strike K')
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Implied vol')
axes[0].legend(fontsize=8)
fig.suptitle('Observed quotes vs nodal and smoothed eSSVI slices', y=1.02)
plt.tight_layout()
plt.show()


,fit_success,price_rmse,max_abs_price_error,node_validation_ok,projection_success,projection_validation_ok,projection_dupire_invalid_count,projection_total_points,projection_message
0,True,0.024943,0.114532,True,True,True,0,930,OK


,T,theta,psi,rho,eta,g_plus,g_minus
0,0.10,0.004312,0.036693,-0.295405,-0.010839,0.025854,0.047532
1,0.15,0.006729,0.045042,-0.232700,-0.010481,0.034561,0.055523
2,0.20,0.009267,0.051073,-0.212092,-0.010832,0.040241,0.061906
3,0.25,0.011919,0.056916,-0.194259,-0.011056,0.045859,0.067972
4,0.33,0.016375,0.066605,-0.172744,-0.011506,0.055099,0.078110
5,0.50,0.026681,0.083020,-0.139740,-0.011601,0.071419,0.094621
6,0.75,0.043876,0.099663,-0.128045,-0.012761,0.086902,0.112425
7,1.00,0.062254,0.118476,-0.099242,-0.011758,0.106718,0.130234
8,1.25,0.082327,0.132620,-0.090303,-0.011976,0.120644,0.144596
9,1.50,0.103637,0.150034,-0.086183,-0.012930,0.137104,0.162965


## 2) Compare quote fit before and after smoothing

The fit tables check that smoothing changes the time structure more than the quote-level agreement on the shared market data.

In [3]:
quote_compare = art.tables['essvi_quote_compare'].copy()
worst_quotes = quote_compare.sort_values('abs_iv_error_bp_smoothed', ascending=False).head(12)
fit_by_T = (
    quote_compare.groupby('T')
    .agg(
        mean_abs_iv_error_bp_smoothed=('abs_iv_error_bp_smoothed', 'mean'),
        max_abs_iv_error_bp_smoothed=('abs_iv_error_bp_smoothed', 'max'),
    )
    .reset_index()
)

display(art.tables['essvi_quote_summary'])
display(worst_quotes)
display(fit_by_T)

plt.figure(figsize=(8, 4))
plt.plot(
    fit_by_T['T'],
    fit_by_T['mean_abs_iv_error_bp_smoothed'],
    marker='o',
    label='mean abs IV error',
)
plt.plot(
    fit_by_T['T'],
    fit_by_T['max_abs_iv_error_bp_smoothed'],
    marker='s',
    label='max abs IV error',
)
plt.xlabel('Expiry T')
plt.ylabel('Error (bp)')
plt.title('Smoothed eSSVI quote fit by expiry')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


,mean_abs_iv_error_bp_nodal,max_abs_iv_error_bp_nodal,mean_abs_iv_error_bp_smoothed,max_abs_iv_error_bp_smoothed,mean_abs_price_error_nodal,max_abs_price_error_nodal,mean_abs_price_error_smoothed,max_abs_price_error_smoothed
0,15.343486,241.681589,15.343486,241.681589,0.016574,0.114532,0.016574,0.114532


,T,x,K,F,iv_obs,iv_true,y,w_obs,w_true,iv_noise_bp,is_call,kind,price_mkt,iv_nodal,iv_smoothed,abs_iv_error_bp_nodal,abs_iv_error_bp_smoothed,price_nodal,price_smoothed,abs_price_error_nodal,abs_price_error_smoothed
0,0.10,-0.30,74.230134,100.200200,0.344461,0.344460,-0.30,0.011865,0.011865,0.012302,False,put,0.008306,0.320293,0.320293,241.681589,241.681589,0.003804,0.003804,0.004502,0.004502
1,0.10,-0.28,75.729682,100.200200,0.334451,0.334152,-0.28,0.011186,0.011166,2.987455,False,put,0.011548,0.313107,0.313107,213.441064,213.441064,0.005957,0.005957,0.005591,0.005591
2,0.10,-0.26,77.259523,100.200200,0.323493,0.323767,-0.26,0.010465,0.010482,-2.741379,False,put,0.015796,0.305792,0.305792,177.002670,177.002670,0.009330,0.009330,0.006466,0.006466
3,0.10,-0.24,78.820269,100.200200,0.312439,0.313330,-0.24,0.009762,0.009818,-8.905918,False,put,0.021808,0.298348,0.298348,140.907606,140.907606,0.014615,0.014615,0.007193,0.007193
31,0.15,-0.30,74.304401,100.300450,0.308949,0.309427,-0.30,0.014317,0.014362,-4.777533,False,put,0.020155,0.296146,0.296146,128.036085,128.036085,0.013731,0.013731,0.006424,0.006424
4,0.10,-0.22,80.412544,100.200200,0.302420,0.302875,-0.22,0.009146,0.009173,-4.546708,False,put,0.031299,0.290775,0.290775,116.450424,116.450424,0.022896,0.022896,0.008404,0.008404
32,0.15,-0.28,75.805450,100.300450,0.300724,0.301702,-0.28,0.013565,0.013654,-9.785191,False,put,0.027173,0.290280,0.290280,104.434152,104.434152,0.020143,0.020143,0.007030,0.007030
34,0.15,-0.24,78.899129,100.300450,0.287287,0.286226,-0.24,0.012380,0.012289,10.608986,False,put,0.054394,0.278353,0.278353,89.341064,89.341064,0.043304,0.043304,0.011090,0.011090
33,0.15,-0.26,77.336821,100.300450,0.293152,0.293961,-0.26,0.012891,0.012962,-8.088372,False,put,0.037539,0.284348,0.284348,88.043752,88.043752,0.029541,0.029541,0.007999,0.007999
62,0.20,-0.30,74.378743,100.400801,0.291483,0.291356,-0.30,0.016992,0.016978,1.272684,False,put,0.040842,0.282986,0.282986,84.970057,84.970057,0.032322,0.032322,0.008521,0.008521


,T,mean_abs_iv_error_bp_smoothed,max_abs_iv_error_bp_smoothed
0,0.10,45.646083,241.681589
1,0.15,34.371999,128.036085
2,0.20,20.167742,84.970057
3,0.25,14.855374,65.785197
4,0.33,10.314169,30.427274
5,0.50,7.278467,23.902352
6,0.75,8.028567,19.448451
7,1.00,7.013693,24.398731
8,1.25,8.059617,25.477642
9,1.50,6.239188,21.860360


## 3) Compare expiry-direction smoothness for the Dupire handoff

The expiry-direction diagnostics show why the smoothed eSSVI object is the cleaner handoff once time derivatives and local-vol extraction matter.

In [4]:
smoothness = art.tables['essvi_time_smoothness_compare'].copy()
smoothness['max_jump_ratio_svi_over_smoothed'] = (
    smoothness['max_abs_wT_jump_svi'] / smoothness['max_abs_wT_jump_smoothed']
)

display(smoothness)
display(art.tables['essvi_localvol_handoff_summary'])
display(art.tables['essvi_localvol_handoff_worst'].head(10))

plt.figure(figsize=(8, 4))
plt.semilogy(
    smoothness['T_knot'],
    smoothness['max_abs_wT_jump_svi'],
    marker='o',
    label='slice-stack SVI',
)
plt.semilogy(
    smoothness['T_knot'],
    smoothness['max_abs_wT_jump_smoothed'],
    marker='s',
    label='smoothed eSSVI',
)
plt.xlabel('Expiry knot T')
plt.ylabel('max |jump in w_T|')
plt.title('Time-smoothness comparison across expiry knots')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


,T_knot,max_abs_wT_jump_svi,mean_abs_wT_jump_svi,max_abs_wT_jump_smoothed,mean_abs_wT_jump_smoothed,max_jump_ratio_svi_over_smoothed
0,0.15,0.080696,0.041167,0.000082,0.000043,988.288624
1,0.20,0.046021,0.025094,0.000027,0.000013,1719.239926
2,0.25,0.050251,0.024871,0.000024,0.000017,2052.203917
3,0.33,0.060117,0.031881,0.000013,0.000008,4612.528670
4,0.50,0.043331,0.025605,0.000012,0.000008,3609.975893
5,0.75,0.030832,0.016916,0.000012,0.000006,2644.112350
6,1.00,0.015677,0.009221,0.000009,0.000006,1751.398179
7,1.25,0.016616,0.008183,0.000009,0.000006,1814.817340
8,1.50,0.013674,0.006932,0.000010,0.000007,1388.958106


,invalid_count,invalid_frac,sigma_min,sigma_median,sigma_max,denom_abs_min
0,0,0.0,0.208672,0.277583,0.30931,0.547313


,rank,T,y,K,denom,local_var,sigma,invalid,reasons
0,1,0.12,-0.20,82.069807,0.547313,0.095672,0.309310,False,
1,2,0.12,-0.18,83.727727,0.574858,0.089414,0.299022,False,
2,3,0.12,-0.16,85.419139,0.606935,0.083252,0.288534,False,
3,4,0.12,-0.14,87.144720,0.644427,0.077219,0.277883,False,
4,5,0.12,0.20,122.433765,0.670392,0.075997,0.275676,False,
5,6,0.12,-0.12,88.905160,0.688291,0.071363,0.267138,False,
6,7,0.12,0.18,120.009414,0.717198,0.069973,0.264524,False,
7,8,0.12,-0.10,90.701164,0.739427,0.065745,0.256407,False,


## Output objects

- `art.nodal_surface`: calibrated eSSVI node surface on the shared quote set
- `art.smoothed_surface`: smoother implied surface used for the Dupire handoff
- `art.localvol`: local-vol surface derived from the smoothed implied surface
- summary tables for projection quality, quote fit, and expiry-direction smoothness

Demo 08 starts from this smoothed implied surface for the local-vol and PDE checks.